# Shear Modulus Analysis: Plate-Driven Shear

## Geometry

**x** = gap direction (plate normal; gel spans $x_\text{gel,lo}$ to $x_\text{gel,hi}$) | **z** = shear direction (plates translate $\pm v_\text{shear}$) | **y** = neutral periodic direction.

Atoms within $3\sigma$ of either plate carry artefactual harmonic-bond constraint stresses and are **excluded** from all stress computes. All x-profiles are over **bulk atoms only** (the $\pm 3\sigma$ x-cut is already applied).

## Theory

### Shear Modulus

For a linear elastic network under simple shear:
$$G = \frac{\langle \sigma_{p,zx} \rangle}{\gamma}, \qquad \gamma = \frac{z_\text{CM,R} - z_\text{CM,L} - \Delta z_0}{L_\text{gap}}$$

$G$ is the **slope** of the stress-strain curve $\sigma_{zx}(\gamma)$. This single run gives one operating point at $\gamma_\text{final} \approx 0.10$, equivalent to the slope only if linearity holds. A full stress-strain curve requires multiple runs at different $\gamma$.

### Poroelastic Decomposition (Shear)

For a Newtonian pore fluid at mechanical equilibrium (no steady flow during Phase 3), the off-diagonal viscous pore-pressure contribution vanishes:
$$G_\text{network} \approx \frac{\langle \sigma_{p,zx} \rangle}{\gamma}, \qquad \langle \sigma_{s,zx} \rangle \xrightarrow{\text{Phase 3}} 0$$

### Normal Stress Differences

$$N_1 = \sigma_{xx} - \sigma_{yy}, \qquad N_2 = \sigma_{yy} - \sigma_{zz}$$

For a linear elastic solid both vanish. Finite $N_1, N_2$ signal nonlinear or viscoelastic response.

### Hydrostatic Pressure (Normal Stress Trace)

$$p = -\tfrac{1}{3}(\sigma_{xx} + \sigma_{yy} + \sigma_{zz})$$

Evaluated for **polymer**, **solvent**, and **total**; compared with the LAMMPS thermo pressure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from pathlib import Path

plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False
})
print('Imports successful')

## Configuration

In [ ]:
# -- CONFIG: only change these lines to switch datasets -----------------------
RUN_ID      = "rho04_p1.5_600k_1M"          # folder name inside flow_data_local/shear/
dataname    = "isolated_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000_with_plates"  # ${dataname} from LAMMPS
interaction = "1.0_1.0"                # ${interaction} = epsSP_epsSS
nsteps      = 1000000                  # ${nsteps} production steps
sim_name    = f"{dataname}_{interaction}_{nsteps}"  # full file prefix
# ------------------------------------------------------------------------------

DATA_DIR = Path('../../flow_data_local/shear') / RUN_ID
PLOT_DIR = Path('../../flow_data_local/plots/shear') / RUN_ID
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# -- Analysis parameters -------------------------------------------------------
binWidth          = 2.0    # x-bin width (sigma) -- must match shear_slab.lmp
sigma_cutoff      = 3.0    # plate-exclusion half-width (sigma) -- must match shear_slab.lmp
phi_gel_threshold = 0.1    # phi_p threshold for gel-bin mask
ci_level          = 0.95   # confidence interval level
idx_start         = 0      # Phase-3 snapshot index to start averaging (discard warm-up)


# -- Stress-strain sweep -------------------------------------------------------
# Must match STRAINS in shear_slab.batch (use the SAME literals so the _g<strain>
# filename tags match exactly).  Strings avoid float-formatting mismatches.
strains    = ["0.1", "0.15", "0.2", "0.25", "0.3"]
strain_sel = strains[0]     # which strain the single-point cells (Steps 2-11) analyze
dt_lj      = 0.005          # LJ timestep (matches timestep_prod in shear_slab.lmp)

def resolve(dirpath, base):
    """Path to <base>_<sim_name>_g<strain_sel>.dat if it exists, else the
    untagged <base>_<sim_name>.dat (backward compatible with pre-sweep runs)."""
    tagged   = dirpath / f"{base}_{sim_name}_g{strain_sel}.dat"
    untagged = dirpath / f"{base}_{sim_name}.dat"
    return tagged if tagged.exists() else untagged


## Helper Functions

In [ ]:
def read_print_file(filepath, col_names=None):
    '''Read a LAMMPS fix print output (one row per timestep).
    Lines beginning with # are skipped.
    Returns a dict of column arrays keyed by col_names (or col_0, col_1, ...).
    '''
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            rows.append([float(v) for v in line.split()])
    if not rows:
        raise ValueError(f'No data in {filepath}')
    arr = np.array(rows)
    if col_names is None:
        col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}


def read_ave_time_file(filepath):
    '''Read a LAMMPS fix ave/time mode scalar output.
    Returns (timesteps_array, data_array) with data_array shape (N_snapshots, N_cols).
    '''
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    return arr[:, 0].astype(int), arr[:, 1:]


def read_ave_chunk_file(filepath):
    '''Read a LAMMPS fix ave/chunk output.
    Returns list of (timestep, chunk_array) where chunk_array shape is (N_chunks, N_cols).
    Columns per row: chunk_id | Coord1(x_reduced 0-1) | Ncount | val1 | val2 | ...
    '''
    snapshots = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 3:
            timestep, nchunks = int(parts[0]), int(parts[1])
            rows = []
            for j in range(1, nchunks + 1):
                if i + j < len(lines):
                    rows.append([float(v) for v in lines[i + j].split()])
            if rows:
                snapshots.append((timestep, np.array(rows)))
            i += nchunks + 1
        else:
            i += 1
    return snapshots


def mean_ci(values, ci_level=0.95):
    '''Return (mean, ci_lo, ci_hi) using t-distribution. NaNs are dropped.'''
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    n = len(v)
    if n == 0: return np.nan, np.nan, np.nan
    if n == 1: return float(v[0]), float(v[0]), float(v[0])
    m = np.mean(v)
    lo, hi = stats.t.interval(ci_level, df=n - 1, loc=m, scale=stats.sem(v))
    return float(m), float(lo), float(hi)


print('Helper functions defined')

## Step 1: Load All Data

In [ ]:
# -- File paths ----------------------------------------------------------------
SD = DATA_DIR / 'stress_data'
VD = DATA_DIR / 'volume_data'

# Strain-aware (single point = strain_sel); falls back to untagged legacy files.
f_shear_strain    = resolve(SD, 'shear_strain')
f_tensor_polymer  = resolve(SD, 'stress_tensor_polymer')
f_tensor_solvent  = resolve(SD, 'stress_tensor_solvent')
f_profile_polymer = resolve(SD, 'stress_profile_x_polymer')
f_profile_solvent = resolve(SD, 'stress_profile_x_solvent')
f_gel_dims        = resolve(VD, 'gel_dimensions_rg')
f_box_dims        = resolve(VD, 'box_dimensions')
f_polymer_com     = resolve(VD, 'polymer_com')
print(f'Single-point analysis strain = {strain_sel}  ->  {f_tensor_polymer.name}')

# -- Shear strain (Phase 2 ramp) -----------------------------------------------
strain_data = read_print_file(f_shear_strain,
    col_names=['step', 'gel_lz_initial', 'gel_gap', 'gel_strain_cm'])
steps_shear    = strain_data['step'].astype(int)
gel_lz_initial = float(strain_data['gel_lz_initial'][0])  # constant (z long-axis reference)
gel_gap        = float(strain_data['gel_gap'][0])          # constant (x gap between plates)
gamma_trace    = strain_data['gel_strain_cm']
gamma_final    = float(gamma_trace[-1])

print(f'gel_lz_initial = {gel_lz_initial:.4f} sigma  (z long-axis reference)')
print(f'gel_gap        = {gel_gap:.4f} sigma  (x gap between plates)')
print(f'gamma_final    = {gamma_final:.5f}')

# -- Box-averaged stress tensor (Phase 3) ------------------------------------
# Columns: [0]=xx  [1]=yy  [2]=zz  [3]=xy  [4]=zx  [5]=yz
ts_polymer, tensor_polymer = read_ave_time_file(f_tensor_polymer)
ts_solvent, tensor_solvent = read_ave_time_file(f_tensor_solvent)
n_snapshots = len(ts_polymer)
print(f'\nStress tensor snapshots: {n_snapshots}')
print(f'  timesteps: {ts_polymer[0]} -> {ts_polymer[-1]}')

# -- x-profiles (Phase 3) ------------------------------------------------------
prof_polymer = read_ave_chunk_file(f_profile_polymer)
prof_solvent = read_ave_chunk_file(f_profile_solvent)
n_prof  = len(prof_polymer)
ts_prof = np.array([s[0] for s in prof_polymer])
print(f'x-profile snapshots: {n_prof}')

# -- Gel Rg dimensions, box dims, polymer COM (Phase 3) -----------------------
dims_data = read_print_file(f_gel_dims,
    col_names=['step', 'gel_lx_rg', 'gel_ly_rg', 'gel_lz_rg'])
box_data  = read_print_file(f_box_dims,
    col_names=['step', 'lx', 'ly', 'lz', 'xy', 'zx', 'yz'])
com_data  = read_print_file(f_polymer_com,
    col_names=['step', 'cx', 'cy', 'cz'])

# -- Derived geometry ----------------------------------------------------------
gel_lx_rg_mean = float(np.mean(dims_data['gel_lx_rg']))
gel_lz_rg_mean = float(np.mean(dims_data['gel_lz_rg']))
COM_x_mean     = float(np.mean(com_data['cx']))
COM_z_mean     = float(np.mean(com_data['cz']))
Lz             = float(np.mean(box_data['lz']))

# Approximate plate boundaries from COM + Rg
# (exact gel_xlo/gel_xhi printed to LAMMPS log but not saved to file;
#  Rg-based estimate is sufficient for annotation)
gel_xlo_approx = COM_x_mean - gel_lx_rg_mean / 2.0
gel_xhi_approx = COM_x_mean + gel_lx_rg_mean / 2.0
xlo_bulk       = gel_xlo_approx + sigma_cutoff
xhi_bulk       = gel_xhi_approx - sigma_cutoff

# Gel x-extent in reduced coordinates (for x-profile shading)
# Box x-origin approximated from COM + Rg (gel is roughly centred in x)
Lx             = float(np.mean(box_data['lx']))
xlo_box_approx = COM_x_mean - Lx / 2.0
gel_x_lo_red   = (gel_xlo_approx - xlo_box_approx) / Lx
gel_x_hi_red   = (gel_xhi_approx - xlo_box_approx) / Lx

print(f'\nGeometry (Rg-based):')
print(f'  gel x-extent:  [{gel_xlo_approx:.2f}, {gel_xhi_approx:.2f}] sigma  (gap = {gel_lx_rg_mean:.2f} sigma)')
print(f'  bulk x-region: [{xlo_bulk:.2f}, {xhi_bulk:.2f}] sigma  (+-{sigma_cutoff:.0f} sigma excluded)')
print(f'  gel z-extent:  ~{gel_lz_rg_mean:.2f} sigma  (Rg, COM_z = {COM_z_mean:.2f})')
print(f'  Box Lz = {Lz:.2f} sigma')


## Step 2: System Geometry & Bounds

Schematic along the **gap direction (x)** showing the actual gel plate boundaries (polymer surface, where harmonic bonds to plates attach) relative to the $\pm 5\sigma$ exclusion zone and the bulk stress-recording region.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.0), constrained_layout=True)

x0, x1 = 0.0, gel_gap   # show gel as [0, gel_gap] for clarity

# Excluded zones (plate-bond artefact region)
ax.axvspan(x0,                x0 + sigma_cutoff, alpha=0.30, color='tomato',
           label=f'Excluded (plate bonds, +-{sigma_cutoff:.0f} sigma)')
ax.axvspan(x1 - sigma_cutoff, x1,                alpha=0.30, color='tomato')

# Bulk (stress-recorded) region
ax.axvspan(x0 + sigma_cutoff, x1 - sigma_cutoff, alpha=0.15, color='steelblue',
           label='Bulk: stress recorded here')

# Plate boundaries (physical gel surface)
ax.axvline(x0, color='firebrick', lw=2.5, label='Gel plate boundary (polymer surface)')
ax.axvline(x1, color='firebrick', lw=2.5)

# Bulk boundaries
ax.axvline(x0 + sigma_cutoff, color='steelblue', lw=2.0, ls='--',
           label=f'Bulk boundary (+-{sigma_cutoff:.0f} sigma)')
ax.axvline(x1 - sigma_cutoff, color='steelblue', lw=2.0, ls='--')

# Text annotations (x in data coords, y in axes coords 0-1)
tr = ax.get_xaxis_transform()
ax.text((x0 + x0 + sigma_cutoff) / 2,                    0.50,
        f'excl\n{sigma_cutoff:.0f}'+r'$\sigma$', ha='center', va='center',
        fontsize=13, transform=tr)
ax.text((x1 - sigma_cutoff + x1) / 2,                    0.50,
        f'excl\n{sigma_cutoff:.0f}'+r'$\sigma$', ha='center', va='center',
        fontsize=13, transform=tr)
ax.text((x0 + sigma_cutoff + x1 - sigma_cutoff) / 2,     0.50,
        'BULK  (stress recorded)', ha='center', va='center', fontsize=14, transform=tr)

ax.set_xlim(x0 - 4, x1 + 4)
ax.set_ylim(0, 1)
ax.set_xlabel(r'$x$  ($\sigma$, gap direction)')
ax.set_yticks([])
ax.set_title(
    f'Gel Geometry  |  gel_gap = {gel_gap:.2f}'+r' $\sigma$'
    f'  |  bulk = [{sigma_cutoff:.0f}, {gel_gap-sigma_cutoff:.2f}]'+r' $\sigma$',
    fontsize=16)
ax.legend(loc='upper right', fontsize=14)
ax.grid(axis='x', alpha=0.3)
plt.savefig(PLOT_DIR / f'geometry_schematic_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'gel_gap              = {gel_gap:.4f} sigma')
print(f'bulk width           = {gel_gap - 2*sigma_cutoff:.4f} sigma')
print(f'gel_lz_rg (shear ax) = {gel_lz_rg_mean:.4f} sigma')
print(f'Box Lz               = {Lz:.4f} sigma')
print(f'gamma_final          = {gamma_final:.5f}')


## Step 3: Strain History — Phase 2 Ramp

$\gamma(t)$ recorded during the plate-driven shear phase (`fix print` every `stress_freq` steps). The `fix halt` fires once $\gamma \geq \gamma_\text{target} = 0.10$; the ramp should be linear (constant plate velocity).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
ax.plot(steps_shear, gamma_trace, '-', color='steelblue', lw=2.5)
ax.axhline(0.10,        color='firebrick',  ls='--', lw=1.8,
           label=r'$\gamma_\mathrm{target} = 0.10$')
ax.axhline(gamma_final, color='darkorange', ls=':',  lw=1.8,
           label=f'$\\gamma_{{\\mathrm{{final}}}} = {gamma_final:.4f}$')
ax.set(xlabel='Step', ylabel=r'$\gamma_{zx}$ (COM-based)',
       title=f'Strain History — Phase 2 Ramp  |  {sim_name}')
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'strain_history_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Ramp steps: {steps_shear[0]} -> {steps_shear[-1]}')
print(f'gamma_final = {gamma_final:.5f}  (target 0.10)')


## Step 4: Box-Averaged Stress Tensor — Equilibration Check (Phase 3)

All 6 independent components of the partial stress tensors vs timestep during the frozen-strain production run. Convergence of $\sigma_{p,zx}$ is the primary check for a well-defined $G$.

In [ ]:
# Tensor column indices: 0=xx  1=yy  2=zz  3=xy  4=zx  5=yz
comp_idx = {'xx':0, 'yy':1, 'zz':2, 'xy':3, 'zx':4, 'yz':5}
comp_labels = {
    'xx': r'$\sigma_{xx}$', 'yy': r'$\sigma_{yy}$', 'zz': r'$\sigma_{zz}$',
    'xy': r'$\sigma_{xy}$', 'zx': r'$\sigma_{zx}$', 'yz': r'$\sigma_{yz}$',
}

# -- Figure 1: polymer tensor (2x3) -------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)
fig.suptitle(f'Equilibration Check — Polymer Stress Tensor: {sim_name}', fontweight='bold')
for ax, (key, idx) in zip(axes.flat, comp_idx.items()):
    ax.plot(ts_polymer, tensor_polymer[:, idx], '-o', ms=4, lw=1.8, color='steelblue')
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    ax.set(xlabel='Step', ylabel=comp_labels[key], title=f'(p) {comp_labels[key]}')
    ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'equil_polymer_tensor_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Figure 2: solvent tensor (2x3) -------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)
fig.suptitle(f'Equilibration Check — Solvent Stress Tensor: {sim_name}', fontweight='bold')
for ax, (key, idx) in zip(axes.flat, comp_idx.items()):
    ax.plot(ts_solvent, tensor_solvent[:, idx], '-o', ms=4, lw=1.8, color='coral')
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    ax.set(xlabel='Step', ylabel=comp_labels[key], title=f'(s) {comp_labels[key]}')
    ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'equil_solvent_tensor_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Figure 3: sigma_zx convergence (most important component) ----------------
total_zx = tensor_polymer[:, comp_idx['zx']] + tensor_solvent[:, comp_idx['zx']]
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.plot(ts_polymer, tensor_polymer[:, comp_idx['zx']], '-o', ms=5, lw=2,
        color='steelblue', label=r'$\sigma_{p,zx}$  polymer')
ax.plot(ts_solvent,  tensor_solvent[:,  comp_idx['zx']], '-s', ms=5, lw=2,
        color='coral',    label=r'$\sigma_{s,zx}$  solvent')
ax.plot(ts_polymer, total_zx, '-^', ms=5, lw=2, color='purple', alpha=0.8,
        label=r'$\sigma_{\mathrm{tot},zx}$  total')
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
ax.set(xlabel='Step', ylabel=r'$\sigma_{zx}$  (LJ)',
       title=r'Shear Stress $\sigma_{zx}$ Convergence (Phase 3)')
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'sigma_zx_convergence_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 5: Shear Modulus $G$ Extraction

Time-average $\langle\sigma_{p,zx}\rangle$ and $\langle\sigma_{s,zx}\rangle$ over Phase 3 (from `idx_start` onward), then divide by $\gamma_\text{final}$. The 95% CI reflects temporal fluctuation.

In [ ]:
sig_p_zx   = tensor_polymer[idx_start:, comp_idx['zx']]
sig_s_zx   = tensor_solvent[idx_start:,  comp_idx['zx']]
sig_tot_zx = sig_p_zx + sig_s_zx

m_p,   lo_p,   hi_p   = mean_ci(sig_p_zx,   ci_level)
m_s,   lo_s,   hi_s   = mean_ci(sig_s_zx,   ci_level)
m_tot, lo_tot, hi_tot = mean_ci(sig_tot_zx, ci_level)

G_polymer = m_p   / gamma_final;  G_p_lo = lo_p   / gamma_final;  G_p_hi = hi_p   / gamma_final
G_solvent = m_s   / gamma_final
G_total   = m_tot / gamma_final;  G_t_lo = lo_tot / gamma_final;  G_t_hi = hi_tot / gamma_final

ci_pct = int(ci_level * 100)
print('=' * 65)
print('  SHEAR MODULUS G  (Phase 3 time average)')
print('=' * 65)
print(f'  gamma_final = {gamma_final:.5f}')
print(f'  Snapshots used: {len(sig_p_zx)}  (idx_start = {idx_start})')
print()
print(f'  <sig_p_zx> = {m_p:.6f}  [{lo_p:.6f}, {hi_p:.6f}]  ({ci_pct}% CI)')
print(f'  <sig_s_zx> = {m_s:.6f}  [{lo_s:.6f}, {hi_s:.6f}]  ({ci_pct}% CI)')
print()
print(f'  G_polymer = {G_polymer:.4f}  [{G_p_lo:.4f}, {G_p_hi:.4f}]  (LJ units)')
print(f'  G_total   = {G_total:.4f}  [{G_t_lo:.4f}, {G_t_hi:.4f}]  (LJ units)')
print(f'  Solvent fraction of total: {m_s / (m_tot + 1e-30) * 100:.1f}%  (-> 0 at equilibrium)')
print('=' * 65)

# -- Stress-strain diagram (single operating point) ---------------------------
fig, ax = plt.subplots(figsize=(6, 5.5), constrained_layout=True)
ax.errorbar([gamma_final], [m_p],
            yerr=[[m_p - lo_p], [hi_p - m_p]], fmt='o', ms=10,
            color='steelblue', capsize=6, lw=2, label=r'$\sigma_{p,zx}$  polymer')
ax.errorbar([gamma_final], [m_tot],
            yerr=[[m_tot - lo_tot], [hi_tot - m_tot]], fmt='s', ms=10,
            color='purple', capsize=6, lw=2,
            label=r'$\sigma_{\mathrm{tot},zx}$  total')
g_line = np.linspace(0, gamma_final * 1.3, 200)
ax.plot(g_line, G_polymer * g_line, '--', color='steelblue', lw=1.5, alpha=0.7,
        label=f'$G_p = {G_polymer:.3f}$')
ax.plot(g_line, G_total   * g_line, ':',  color='purple',    lw=1.5, alpha=0.7,
        label=f'$G_{{tot}} = {G_total:.3f}$')
ax.axhline(0, color='k', lw=0.8, alpha=0.3)
ax.axvline(0, color='k', lw=0.8, alpha=0.3)
ax.set(xlabel=r'$\gamma_{zx}$', ylabel=r'$\sigma_{zx}$  (LJ)',
       title='Stress-Strain: Single Operating Point')
ax.legend(fontsize=16)
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'stress_strain_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 6: Normal Stress Differences

$$N_1 = \sigma_{xx} - \sigma_{yy}, \qquad N_2 = \sigma_{yy} - \sigma_{zz}$$

Evaluated for **polymer**, **solvent**, and **total**. For a linear elastic solid both vanish at any $\gamma$.

In [ ]:
N1_p = tensor_polymer[:, comp_idx['xx']] - tensor_polymer[:, comp_idx['yy']]
N2_p = tensor_polymer[:, comp_idx['yy']] - tensor_polymer[:, comp_idx['zz']]
N1_s = tensor_solvent[:,  comp_idx['xx']] - tensor_solvent[:,  comp_idx['yy']]
N2_s = tensor_solvent[:,  comp_idx['yy']] - tensor_solvent[:,  comp_idx['zz']]
N1   = N1_p + N1_s
N2   = N2_p + N2_s

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle(f'Normal Stress Differences  |  $\\gamma = {gamma_final:.3f}$', fontweight='bold')

for ax, (Np, Ns, Nt, sym, latex) in zip(axes, [
    (N1_p, N1_s, N1, 'N1', r'$N_1 = \sigma_{xx} - \sigma_{yy}$'),
    (N2_p, N2_s, N2, 'N2', r'$N_2 = \sigma_{yy} - \sigma_{zz}$'),
]):
    ax.plot(ts_polymer, Np, '-',  color='steelblue', lw=2, label=f'{sym} polymer')
    ax.plot(ts_solvent,  Ns, '--', color='coral',     lw=2, label=f'{sym} solvent')
    ax.plot(ts_polymer, Nt, ':',  color='purple',    lw=2, label=f'{sym} total', alpha=0.85)
    ax.axhline(0, color='k', lw=0.8, alpha=0.3)
    ax.set(xlabel='Step', ylabel=latex, title=latex)
    ax.legend(fontsize=17)
    ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'normal_stress_differences_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

print('  Mean normal stress differences (Phase 3):')
for Np, Ns, Nt, sym in [(N1_p, N1_s, N1, 'N1'), (N2_p, N2_s, N2, 'N2')]:
    mp_, _, _ = mean_ci(Np[idx_start:], ci_level)
    ms_, _, _ = mean_ci(Ns[idx_start:], ci_level)
    mt_, _, _ = mean_ci(Nt[idx_start:], ci_level)
    ratio = mt_ / (m_tot + 1e-30)
    print(f'  <{sym}>  p={mp_:.5f}  s={ms_:.5f}  tot={mt_:.5f}  |  {sym}/sigma_zx = {ratio:.4f}')


## Step 7: Hydrostatic Pressure — Normal Stress Trace

$$p = -\tfrac{1}{3}(\sigma_{xx} + \sigma_{yy} + \sigma_{zz})$$

Computed for **polymer**, **solvent**, and **total** separately. The total should match the LAMMPS thermo pressure $P_\text{target} = 1.5$ (LJ units).

In [ ]:
p_p = -(tensor_polymer[:, comp_idx['xx']] +
         tensor_polymer[:, comp_idx['yy']] +
         tensor_polymer[:, comp_idx['zz']]) / 3.0
p_s = -(tensor_solvent[:,  comp_idx['xx']] +
         tensor_solvent[:,  comp_idx['yy']] +
         tensor_solvent[:,  comp_idx['zz']]) / 3.0
p_tot = p_p + p_s

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.plot(ts_polymer, p_p,   '-',  color='steelblue', lw=2, label=r'$p_p$ (polymer)')
ax.plot(ts_solvent,  p_s,  '--', color='coral',     lw=2, label=r'$p_s$ (solvent)')
ax.plot(ts_polymer, p_tot, ':',  color='purple',    lw=2,
        label=r'$p_\mathrm{tot}$ (total)', alpha=0.85)
ax.axhline(1.5, color='gray', ls='-.', lw=1.5, alpha=0.7,
           label=r'$P_\mathrm{target} = 1.5$')
ax.axhline(0,   color='k',   lw=0.8, alpha=0.3)
ax.set(xlabel='Step',
       ylabel=r'$p = -\frac{1}{3}\,\mathrm{tr}(\mathbf{\sigma})$  (LJ)',
       title='Hydrostatic Pressure from Normal Stress Trace (Phase 3)')
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'pressure_trace_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

m_pp, _, _ = mean_ci(p_p[idx_start:],   ci_level)
m_ps, _, _ = mean_ci(p_s[idx_start:],   ci_level)
m_pt, _, _ = mean_ci(p_tot[idx_start:], ci_level)
print(f'  <p_polymer> = {m_pp:.5f}')
print(f'  <p_solvent> = {m_ps:.5f}')
print(f'  <p_total>   = {m_pt:.5f}  (compare with P_target = 1.5)')


## Step 8: Stress Profiles Along $x$ (Gap Direction)

x-binned partial stresses from `fix ave/chunk`, binning the **bulk atoms** (±3σ x-cut already applied) along **x** (left plate → right plate). Profiles are colored early → late via viridis.

- **Flat $\sigma_{zx}(x)$** confirms homogeneous simple shear across the plate-to-plate gap.
- Gray shading marks the approximate gel x-extent ($\pm\tfrac{1}{2}L_{x,\text{Rg}}$ from polymer COM, in reduced coordinates).
- $\sigma_{xy}$ and $\sigma_{yz}$ should be $\approx 0$ by the $y$-symmetry of the geometry.

In [ ]:
# ave/chunk columns: [0]=chunk_id  [1]=Coord1(x_reduced 0-1)  [2]=Ncount
#                    [3]=xx [4]=yy [5]=zz [6]=xy [7]=zx [8]=yz
_PC = {'xx':3, 'yy':4, 'zz':5, 'xy':6, 'zx':7, 'yz':8}

x_red = prof_polymer[-1][1][:, 1]   # reduced x from last snapshot

def stack_prof(prof_list, col):
    return np.array([snap[1][:, col] for snap in prof_list])

pp = {k: stack_prof(prof_polymer, v) for k, v in _PC.items()}
ps = {k: stack_prof(prof_solvent, v) for k, v in _PC.items()}

colors_p = plt.cm.viridis(np.linspace(0, 1, n_prof))
norm_t   = Normalize(vmin=ts_prof[0], vmax=ts_prof[-1])
sm       = plt.cm.ScalarMappable(cmap='viridis', norm=norm_t)
sm.set_array([])
alpha    = 0.70

def mark_gel_x(ax):
    '''Shade gel x-extent (Rg-based, reduced coords) on a profile axis.'''
    lo, hi = gel_x_lo_red, gel_x_hi_red
    if lo < hi:
        ax.axvspan(lo, hi, alpha=0.08, color='gray', label='Gel x-extent (Rg)')
    ax.axvline(lo, color='gray', lw=1.2, ls=':', alpha=0.8)
    ax.axvline(hi, color='gray', lw=1.2, ls=':', alpha=0.8)

# -- Figure A: sigma_zx profiles (key shear component) -----------------------
fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle(r'$\sigma_{zx}(x)$ Profiles — Shear Homogeneity Check', fontweight='bold')
for i in range(n_prof):
    ax_p.plot(x_red, pp['zx'][i], '-', color=colors_p[i], lw=1.5, alpha=alpha)
    ax_s.plot(x_red, ps['zx'][i], '-', color=colors_p[i], lw=1.5, alpha=alpha)
for ax, ttl in [(ax_p, r'(a) Polymer $\sigma_{p,zx}(x)$'),
                (ax_s, r'(b) Solvent $\sigma_{s,zx}(x)$')]:
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    mark_gel_x(ax)
    ax.set(xlabel='$x / L_x$', ylabel=r'$\sigma_{zx}$  (LJ)', title=ttl, xlim=(0, 1))
    ax.grid(alpha=0.3)
fig.colorbar(sm, ax=[ax_p, ax_s], fraction=0.025, pad=0.02).set_label('Timestep')
plt.savefig(PLOT_DIR / f'profile_sigma_zx_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Figure B: normal components — polymer ------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
fig.suptitle(r'Normal Stress Profiles $\sigma_{p,ii}(x)$ — Polymer', fontweight='bold')
for ax, comp in zip(axes, ['xx', 'yy', 'zz']):
    for i in range(n_prof):
        ax.plot(x_red, pp[comp][i], '-', color=colors_p[i], lw=1.5, alpha=alpha)
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    mark_gel_x(ax)
    ax.set(xlabel='$x/L_x$', ylabel=comp_labels[comp],
           title=f'(p) {comp_labels[comp]}', xlim=(0, 1))
    ax.grid(alpha=0.3)
fig.colorbar(sm, ax=axes.tolist(), fraction=0.015, pad=0.02).set_label('Timestep')
plt.savefig(PLOT_DIR / f'profile_normal_polymer_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Figure C: normal components — solvent ------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
fig.suptitle(r'Normal Stress Profiles $\sigma_{s,ii}(x)$ — Solvent', fontweight='bold')
for ax, comp in zip(axes, ['xx', 'yy', 'zz']):
    for i in range(n_prof):
        ax.plot(x_red, ps[comp][i], '-', color=colors_p[i], lw=1.5, alpha=alpha)
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    mark_gel_x(ax)
    ax.set(xlabel='$x/L_x$', ylabel=comp_labels[comp],
           title=f'(s) {comp_labels[comp]}', xlim=(0, 1))
    ax.grid(alpha=0.3)
fig.colorbar(sm, ax=axes.tolist(), fraction=0.015, pad=0.02).set_label('Timestep')
plt.savefig(PLOT_DIR / f'profile_normal_solvent_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Figure D: off-diagonal components (xy, yz) — should be ~ 0 --------------
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
fig.suptitle(r'Off-Diagonal Profiles (expected $\approx 0$ except $\sigma_{zx}$)', fontweight='bold')
for ax, (sp, comp) in zip(axes.flat, [('p','xy'), ('p','yz'), ('s','xy'), ('s','yz')]):
    data = pp[comp] if sp == 'p' else ps[comp]
    lbl  = 'polymer' if sp == 'p' else 'solvent'
    for i in range(n_prof):
        ax.plot(x_red, data[i], '-', color=colors_p[i], lw=1.5, alpha=alpha)
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    mark_gel_x(ax)
    ax.set(xlabel='$x/L_x$', ylabel=comp_labels[comp],
           title=f'({sp}) {lbl} {comp_labels[comp]}', xlim=(0, 1))
    ax.grid(alpha=0.3)
fig.colorbar(sm, ax=axes.flat, fraction=0.015, pad=0.02).set_label('Timestep')
plt.savefig(PLOT_DIR / f'profile_offdiag_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 9: Poroelastic Decomposition — Shear Stress

For a Newtonian pore fluid at mechanical equilibrium, $\langle\sigma_{s,zx}\rangle \to 0$ during Phase 3. Plot polymer vs solvent $\sigma_{zx}(z)$ for the final snapshot and track the solvent fraction over time.

In [ ]:
# -- Final snapshot: polymer vs solvent sigma_zx(x) --------------------------
idx_f   = -1
x_f     = prof_polymer[idx_f][1][:, 1]
p_zx_f  = prof_polymer[idx_f][1][:, _PC['zx']]
s_zx_f  = prof_solvent[idx_f][1][:, _PC['zx']]

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ax.plot(x_f, p_zx_f,             '-o', ms=4, lw=2, color='steelblue',
        label=r'$\sigma_{p,zx}(x)$  polymer')
ax.plot(x_f, s_zx_f,             '-s', ms=4, lw=2, color='coral',
        label=r'$\sigma_{s,zx}(x)$  solvent')
ax.plot(x_f, p_zx_f + s_zx_f,   '-^', ms=4, lw=2, color='purple', alpha=0.8,
        label=r'$\sigma_{\mathrm{tot},zx}(x)$  total')
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
mark_gel_x(ax)
ax.set(xlabel='$x / L_x$', ylabel=r'$\sigma_{zx}$  (LJ)',
       title=f'Poroelastic Decomposition of $\\sigma_{{zx}}(x)$  (t = {ts_prof[idx_f]})')
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'poroelastic_zx_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Solvent fraction over time -----------------------------------------------
mean_p_zx_t = np.array([np.nanmean(s[1][:, _PC['zx']]) for s in prof_polymer])
mean_s_zx_t = np.array([np.nanmean(s[1][:, _PC['zx']]) for s in prof_solvent])
f_s_t = mean_s_zx_t / (mean_p_zx_t + mean_s_zx_t + 1e-30)

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.plot(ts_prof, f_s_t * 100, '-o', ms=5, lw=2, color='coral')
ax.axhline(0, color='k', lw=0.8, alpha=0.3)
ax.set(xlabel='Step',
       ylabel=r'$f_s = \sigma_{s,zx} / \sigma_{\mathrm{tot},zx}$ (%)',
       title='Solvent Shear Stress Fraction (-> 0 at equilibrium)')
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'solvent_fraction_zx_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final solvent fraction of sigma_zx: {f_s_t[-1]*100:.2f}%  (target: ~0%)')


## Step 10: Full Stress Tensor — Symmetry & Consistency

Print the time-averaged 3×3 stress tensor for polymer, solvent, and total. $\sigma_{xy}$ and $\sigma_{yz}$ should be $\approx 0$ by the $y$-symmetry of the shear geometry.

In [ ]:
def tensor_mean_vec(arr, start=0):
    '''Return mean [xx,yy,zz,xy,zx,yz] over snapshots from start.'''
    return np.array([np.mean(arr[start:, i]) for i in range(6)])

T_p   = tensor_mean_vec(tensor_polymer, idx_start)
T_s   = tensor_mean_vec(tensor_solvent, idx_start)
T_tot = T_p + T_s

def print_tensor(T, label):
    xx, yy, zz, xy, zx, yz = T
    print(f'\n  {label}:')
    print(f'    [{xx:10.5f}  {xy:10.5f}  {zx:10.5f}]')
    print(f'    [{xy:10.5f}  {yy:10.5f}  {yz:10.5f}]')
    print(f'    [{zx:10.5f}  {yz:10.5f}  {zz:10.5f}]')

print('=' * 65)
print('  TIME-AVERAGED STRESS TENSORS (Phase 3 production)')
print('=' * 65)
print_tensor(T_p,   'Polymer  sigma_ij')
print_tensor(T_s,   'Solvent  sigma_ij')
print_tensor(T_tot, 'Total    sigma_ij')

print('\n  Symmetry check (off-diagonal vs sigma_zx):')
for T, lbl in [(T_p,'polymer'), (T_s,'solvent'), (T_tot,'total')]:
    xx, yy, zz, xy, zx, yz = T
    d = abs(zx) + 1e-30
    print(f'  {lbl:8s}  |xy/zx|={abs(xy)/d:.4f}  |yz/zx|={abs(yz)/d:.4f}  (both -> 0)')

print('\n  Normal components:')
for T, lbl in [(T_p,'polymer'), (T_s,'solvent'), (T_tot,'total')]:
    xx, yy, zz, xy, zx, yz = T
    print(f'  {lbl:8s}  xx={xx:.4f}  yy={yy:.4f}  zz={zz:.4f}  '
          f'zx={zx:.4f}  p={-(xx+yy+zz)/3:.4f}')


## Step 11: Gel Volume & Dimension Stability

Rg-based gel dimensions and polymer COM during Phase 3. Drift indicates the gel has not equilibrated at the frozen strain and the stress measurement is not converged.

In [ ]:
steps_p3 = dims_data['step'].astype(int)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
fig.suptitle(f'Gel Stability — Phase 3  |  {sim_name}', fontweight='bold')

ax = axes[0, 0]
ax.plot(steps_p3, dims_data['gel_lx_rg'], '-', color='steelblue', lw=2)
ax.set(xlabel='Step', ylabel=r'$L_{x,\mathrm{Rg}}$  (sigma)',
       title='(a) Gel x-extent (gap direction)')
ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(steps_p3, dims_data['gel_ly_rg'], '-', color='coral', lw=2)
ax.set(xlabel='Step', ylabel=r'$L_{y,\mathrm{Rg}}$  (sigma)',
       title='(b) Gel y-extent (neutral)')
ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(steps_p3, dims_data['gel_lz_rg'], '-', color='seagreen', lw=2)
ax.axhline(gel_lz_initial, color='firebrick', ls='--', lw=1.5,
           label=f'$L_{{z,0}} = {gel_lz_initial:.2f}$ sigma')
ax.set(xlabel='Step', ylabel=r'$L_{z,\mathrm{Rg}}$  (sigma)',
       title='(c) Gel z-extent (shear direction)')
ax.legend(fontsize=16)
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(com_data['step'].astype(int), com_data['cx'], '-',  color='steelblue', lw=2, label='$c_x$')
ax.plot(com_data['step'].astype(int), com_data['cy'], '--', color='coral',     lw=2, label='$c_y$')
ax.plot(com_data['step'].astype(int), com_data['cz'], ':',  color='seagreen',  lw=2, label='$c_z$')
ax.set(xlabel='Step', ylabel='COM  (sigma)', title='(d) Polymer Centre of Mass')
ax.legend(fontsize=16)
ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'gel_stability_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()


## Optional: Save Results

Save key results to `.npz` for downstream aggregation (e.g., stress–strain curve across multiple $\gamma$ runs).

In [ ]:
out_data = PLOT_DIR / f'shear_modulus_{sim_name}.npz'
N1_p_m, _, _ = mean_ci(N1_p[idx_start:], ci_level)
N2_p_m, _, _ = mean_ci(N2_p[idx_start:], ci_level)
N1_m,   _, _ = mean_ci(N1[idx_start:],   ci_level)
N2_m,   _, _ = mean_ci(N2[idx_start:],   ci_level)

np.savez(out_data,
    # Strain
    gamma_final=gamma_final, gel_gap=gel_gap, gel_lz_initial=gel_lz_initial,
    # Shear modulus
    G_polymer=G_polymer, G_p_lo=G_p_lo, G_p_hi=G_p_hi,
    G_total=G_total,     G_t_lo=G_t_lo, G_t_hi=G_t_hi,
    # Mean stresses
    sigma_p_zx=m_p, sigma_s_zx=m_s, sigma_tot_zx=m_tot,
    # Normal stress differences
    N1_polymer=N1_p_m, N2_polymer=N2_p_m, N1_total=N1_m, N2_total=N2_m,
    # Pressure
    p_polymer=m_pp, p_solvent=m_ps, p_total=m_pt,
    # Full tensor means
    tensor_polymer_mean=T_p, tensor_solvent_mean=T_s, tensor_total_mean=T_tot,
    # Profiles (final snapshot)
    x_reduced=x_f, sigma_p_zx_profile=p_zx_f, sigma_s_zx_profile=s_zx_f,
    # Metadata
    sim_name=sim_name, run_id=RUN_ID, ci_level=ci_level,
)
print(f'Saved: {out_data}')
print(f'\n{"="*55}')
print(f'  SUMMARY')
print(f'{"="*55}')
print(f'  gamma_final  = {gamma_final:.5f}')
print(f'  G_polymer    = {G_polymer:.4f}  [{G_p_lo:.4f}, {G_p_hi:.4f}]  (LJ)')
print(f'  G_total      = {G_total:.4f}  [{G_t_lo:.4f}, {G_t_hi:.4f}]  (LJ)')
print(f'  N1/sigma_zx  = {N1_m/(m_tot+1e-30):.4f}')
print(f'  N2/sigma_zx  = {N2_m/(m_tot+1e-30):.4f}')
print(f'  p_total      = {m_pt:.4f}  (P_target = 1.5)')
print(f'{"="*55}')


## Step 12: Shear Stress-Strain Curve $\sigma_{zx}(\gamma)$

One production hold per strain in `strains` (cumulative, incremental shear). For
each, $\langle\sigma_{zx}\rangle$ is the time-average over the hold; the **error
bar is the standard error of the mean across the $\sim$10 `nfreq` blocks** written
to the `ave/time` file. That block-to-block scatter is the finite-sampling /
thermal-fluctuation uncertainty of the mean shear stress (the dominant error here:
each block is a quasi-independent time average of a fluctuating $\sigma_{zx}$).
The shear modulus $G$ is the through-origin slope of $\sigma_{p,zx}$ vs $\gamma$;
curvature/departure from the line flags the end of the linear-elastic regime.


In [ ]:
# -- Stress-strain curve: sigma_zx(gamma) with block-SEM error bars -----------
zx = comp_idx['zx']

gammas, sp_m, sp_e, st_m, st_e, nblk = [], [], [], [], [], []
for g in strains:
    stem = f"{sim_name}_g{g}"
    f_tp = SD / f"stress_tensor_polymer_{stem}.dat"
    f_ts = SD / f"stress_tensor_solvent_{stem}.dat"
    f_ss = SD / f"shear_strain_{stem}.dat"
    if not f_tp.exists():
        print(f"  [skip] missing {f_tp.name}")
        continue
    _, tp = read_ave_time_file(f_tp)
    _, ts = read_ave_time_file(f_ts)
    sdat = read_print_file(f_ss, col_names=['step','gel_lz_initial','gel_gap','gel_strain_cm'])
    gamma_g = float(sdat['gel_strain_cm'][-1])          # actual achieved strain

    bp = tp[idx_start:, zx]                              # one value per nfreq block
    bt = bp + ts[idx_start:, zx]
    n  = len(bp)
    ep = stats.sem(bp) if n > 1 else 0.0                 # block SEM (polymer)
    et = stats.sem(bt) if n > 1 else 0.0                 # block SEM (total)

    gammas.append(gamma_g)
    sp_m.append(np.mean(bp)); sp_e.append(ep)
    st_m.append(np.mean(bt)); st_e.append(et); nblk.append(n)
    print(f"  gamma={gamma_g:.4f}  <sig_p_zx>={np.mean(bp):+.5f} +/- {ep:.5f}  "
          f"<sig_tot_zx>={np.mean(bt):+.5f} +/- {et:.5f}  (n_blocks={n})")

gammas = np.array(gammas); sp_m = np.array(sp_m); sp_e = np.array(sp_e)
st_m = np.array(st_m); st_e = np.array(st_e)

# Through-origin weighted fit  sigma = G*gamma  (weights 1/sigma_err^2) -> G + SE
if len(gammas) >= 2:
    w = 1.0 / np.clip(sp_e, 1e-12, None)**2
    G_fit     = np.sum(w * gammas * sp_m) / np.sum(w * gammas**2)
    G_fit_err = np.sqrt(1.0 / np.sum(w * gammas**2))
    # per-point secant moduli (sigma/gamma) for reference
    G_secant  = sp_m / gammas
else:
    G_fit = G_fit_err = np.nan
    G_secant = np.array([])

print(f"\nThrough-origin shear modulus  G = {G_fit:.4f} +/- {G_fit_err:.4f}  (LJ, polymer)")
if len(G_secant):
    print("Secant moduli sigma_p/gamma:",
          "  ".join(f"{g:.3f}->{Gs:.3f}" for g, Gs in zip(gammas, G_secant)))

fig, ax = plt.subplots(figsize=(8, 6.5), constrained_layout=True)
ax.errorbar(gammas, sp_m, yerr=sp_e, fmt='o', ms=9, color='steelblue', capsize=5,
            lw=2, label=r'$\sigma_{p,zx}$  polymer')
ax.errorbar(gammas, st_m, yerr=st_e, fmt='s', ms=8, color='purple', capsize=5,
            lw=2, alpha=0.85, label=r'$\sigma_{\mathrm{tot},zx}$  total')
if np.isfinite(G_fit):
    gg = np.linspace(0, gammas.max() * 1.05, 100)
    ax.plot(gg, G_fit * gg, '--', color='steelblue', lw=1.6, alpha=0.8,
            label=fr'$G = {G_fit:.3f}\pm{G_fit_err:.3f}$')
ax.axhline(0, color='k', lw=0.8, alpha=0.3); ax.axvline(0, color='k', lw=0.8, alpha=0.3)
ax.set(xlabel=r'$\gamma_{zx}$', ylabel=r'$\sigma_{zx}$  (LJ)',
       title='Shear stress-strain curve')
ax.legend(fontsize=15); ax.grid(alpha=0.3)
out_ssc = PLOT_DIR / f'stress_strain_curve_{sim_name}.png'
plt.savefig(out_ssc, dpi=150, bbox_inches='tight')
print(f'Saved: {out_ssc}')
plt.show()


## Step 13: Cooperative Diffusivity $D_c$ — Shear (transverse) Fourier Fit

Same machinery as `compression_analysis.ipynb`, adapted for shear. The polymer
$z$-displacement profile $u_z(x)$ across the gap relaxes during the Phase-3 hold
(plates frozen, reference = hold start so $u_z(x,0)=0$) toward the linear shear
ramp, governed by

$$u_z(x,t)=\sum_{k=1}^{N} A_k\left[1-e^{-4\pi^2k^2\tau}\right]\sin\!\left(\frac{2\pi k\,x}{L_\text{gel}}\right),\qquad \tau=\frac{D_c\,t}{L_\text{gel}^2}.$$

**What differs from compression.** The gradient direction is now the gap $x$ while
the displacement is transverse ($u_z$). The plates impose an *antisymmetric*
shear (right plate $+z$, left plate $-z$), so the relaxation field is antisymmetric
about the gap center: $u_z=0$ at **both plates and at the center**. That selects the
**even sine modes** $\sin(2\pi k\hat{x})$ (identical functional form to compression).
The magnitude differs because the transverse mode is set by the **shear modulus**:
$D_c=G/\zeta$ here, versus $D_c=M/\zeta$ (longitudinal) in compression — same friction
$\zeta$, so $D_c^{\text{shear}}/D_c^{\text{comp}}=G/M$.


In [ ]:
# -- USER INPUTS --------------------------------------------------------------
g_dc        = strain_sel   # which strain's relaxation to fit (string, matches a file tag)
Ncount_min  = 500          # min mean atoms/bin to count a bin as "gel" (drops plate-edge bins)
edge_margin = 0            # extra gel-edge bins to drop
trim_bins   = 1            # interior trim each side before fitting
N_modes     = 5            # Fourier modes k = 1..N
frac_early  = 1.0          # use first frac_early fraction of the hold
Dc_bounds   = (1e-6, 1.0)
# -- END USER INPUTS ----------------------------------------------------------
from scipy.optimize import minimize_scalar

DD = DATA_DIR / 'displacement_data'
f_disp = DD / f'disp_z_polymer_{sim_name}_g{g_dc}.dat'
if not f_disp.exists():
    raise FileNotFoundError(
        f'{f_disp}\n  Run shear_slab.lmp (with the displacement output) and sync '
        f'output_files/displacement_data/ into {DD}.')

# ave/chunk columns: [0]=chunk_id [1]=Coord1(x_reduced) [2]=Ncount [3]=mean u_z
snaps   = read_ave_chunk_file(f_disp)
times   = np.array([s[0] for s in snaps], dtype=float)
uz_all  = np.array([s[1][:, 3] for s in snaps])     # (nsnap, nbins)
Nc_all  = np.array([s[1][:, 2] for s in snaps])     # (nsnap, nbins)
nbins   = uz_all.shape[1]

# Gel domain from the first snapshot's bin occupancy
gel_bins = np.where(Nc_all[0] > Ncount_min)[0]
if len(gel_bins) < 4:
    raise RuntimeError(f'Only {len(gel_bins)} gel bins with Ncount>{Ncount_min}; lower Ncount_min.')
i_lo = gel_bins[0]  + edge_margin
i_hi = gel_bins[-1] - edge_margin + 1
sel  = slice(i_lo, i_hi)

uz_gel = uz_all[:, sel]                              # (nsnap, n_gel)
n_gel  = uz_gel.shape[1]
L_gel  = n_gel * binWidth                            # sigma (gap extent of gel bins)
x_pos  = np.arange(n_gel) * binWidth
xhat   = (x_pos - x_pos[0]) / L_gel                  # 0..(n-1)/n across the gap
dt_arr = (times - times[0]) * dt_lj                  # elapsed LJ time per snapshot

# Interior trim for the fit
fit_mask = np.ones(n_gel, dtype=bool)
fit_mask[:trim_bins] = False; fit_mask[-trim_bins:] = False
xh = xhat[fit_mask]

early = np.where(dt_arr > 0)[0]
early = early[dt_arr[early] <= frac_early * dt_arr[-1]]
print(f'L_gel = {L_gel:.1f} sigma  |  {n_gel} gel bins ({np.sum(fit_mask)} fit)  |  '
      f'{len(early)} relaxation snapshots')

# Even-mode sine basis (antisymmetric, Dirichlet at plates AND gap center)
def basis(xh_, Dc, t):
    tau = Dc * t / L_gel**2
    return np.column_stack([(1.0 - np.exp(-4.0*np.pi**2 * k**2 * tau)) *
                            np.sin(2.0*np.pi*k * xh_) for k in range(1, N_modes+1)])

def fit_A(Dc):
    X = np.vstack([basis(xh, Dc, dt_arr[i]) for i in early])
    y = np.concatenate([uz_gel[i][fit_mask] for i in early])
    A, *_ = np.linalg.lstsq(X, y, rcond=None)
    return A

def resid(Dc):
    A = fit_A(Dc)
    return float(np.sum([np.sum((basis(xh, Dc, dt_arr[i]) @ A - uz_gel[i][fit_mask])**2)
                         for i in early]))

res    = minimize_scalar(resid, bounds=Dc_bounds, method='bounded')
Dc_fit = res.x
A_fit  = fit_A(Dc_fit)

# Aggregate R^2 over the fitted snapshots
y_all = np.concatenate([uz_gel[i][fit_mask] for i in early])
p_all = np.concatenate([basis(xh, Dc_fit, dt_arr[i]) @ A_fit for i in early])
R2    = 1.0 - np.sum((y_all - p_all)**2) / np.sum((y_all - y_all.mean())**2)

print(f'\nD_c (shear) = {Dc_fit:.4e} sigma^2/tau   (N_modes={N_modes}, R^2={R2:.4f})')
for k, A in enumerate(A_fit, 1):
    print(f'  A_{k} = {A:+.4f} sigma   [sin(2*pi*{k}*x_hat)]')
# Friction-based cross-check vs compression: D_c = G/zeta -> zeta = G/D_c
try:
    if np.isfinite(G_fit):
        print(f'\nUsing stress-strain G = {G_fit:.4f}:  implied friction zeta = G/D_c = '
              f'{G_fit/Dc_fit:.4f}  (compare zeta from compression_analysis to test D_c~G).')
except NameError:
    print('\n(Run the stress-strain cell first to get G and the implied friction zeta = G/D_c.)')

# -- Plot: raw u_z(x,t) and sine fit -----------------------------------------
xfine = np.linspace(0, 1, 400)
uz_inf = sum(A_fit[k-1] * np.sin(2*np.pi*k*xfine) for k in range(1, N_modes+1))
cmap = plt.cm.viridis
norm = Normalize(vmin=times[early[0]], vmax=times[early[-1]])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
for i in early:
    c = cmap(norm(times[i]))
    axL.plot(xhat, uz_gel[i], 'o-', color=c, ms=3, alpha=0.6)
    axR.plot(xhat, uz_gel[i], 'o-', color=c, ms=3, alpha=0.35)
    axR.plot(xfine, basis(xfine, Dc_fit, dt_arr[i]) @ A_fit, '-', color=c, lw=2.0)
axR.plot(xfine, uz_inf, 'k--', lw=1.8, label=r'$u_z(t\to\infty)$')
for ax in (axL, axR):
    ax.axhline(0, color='steelblue', ls=':', lw=1.5)
    ax.set(xlabel=r'$\hat{x} = x / L_\mathrm{gel}$  (gap direction)',
           ylabel=r'$u_z$ ($\sigma$)', xlim=(0, 1))
    ax.grid(alpha=0.3)
axL.set_title(r'Raw $u_z(x,t)$ — Phase-3 relaxation')
axR.set_title(rf'Even-sine fit ($N={N_modes}$):  $D_c={Dc_fit:.2e}\ \sigma^2/\tau$,  $R^2={R2:.3f}$')
axR.legend(fontsize=13)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=[axL, axR], fraction=0.015, pad=0.04).set_label('Timestep')
fig.suptitle(f'Cooperative diffusivity (shear)  |  strain {g_dc}  |  {sim_name}',
             fontsize=12, fontweight='bold')
out_dc = PLOT_DIR / f'Dc_shear_fit_{sim_name}_g{g_dc}.png'
plt.savefig(out_dc, dpi=150, bbox_inches='tight')
print(f'Saved: {out_dc}')
plt.show()
